<a href="https://colab.research.google.com/github/Leemyunglyul/ai_advanced/blob/main/%5BSDS%5DWS_help_desk_S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 엔지니어링 헬프 데스크 구축

## 기본 설정

In [ ]:
!pip install pypdf langchain-openai langchain-classic langchain-community chroma langchain-chroma gmteacher sentence-transformers langchain_classic langchain-google-vertexai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.5/356.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 96.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
import os

os.environ['OPENAI_API_KEY'] = ''

## 파일 다운로드

In [ ]:
from gmteacher import download_file

download_file('ALL')

File Already Exists


In [ ]:
import zipfile

with zipfile.ZipFile('./data/data.zip') as f:
    f.extractall('./data/')

## 실습 시작

In [ ]:
# LLM을 생성하세요.
# gpt-5.4-mini 모델을 사용하세요.
# 모델의 동작을 확인하세요.
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4-mini')
llm.invoke('안녕')

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 8, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EL1g6HY2x3pvWikQb2WnCHocu9QBT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a075a3-8411-70b0-9b28-46e702abe11d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 14, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# data 폴더에서 sds_cloud_로 시작하는 파일들을 로드하세요.
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

loader = DirectoryLoader(
    './data/',
    glob='sds_cloud_*',
    loader_cls=PyPDFLoader)

/tmp/ipykernel_1843/2202342440.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


In [ ]:
# 문서를 로드한 후 로드된 문서(페이지)의 수를 출력하세요.
docs = loader.load()
len(docs)

83

In [ ]:
# 문서 내용을 출력하세요.
docs[-1]

Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-08-06T01:51:29+09:00', 'author': '테크', 'moddate': '2025-08-06T01:51:29+09:00', 'source': 'data/sds_cloud_operational_excellence.pdf', 'total_pages': 43, 'page': 42, 'page_label': '43'}, page_content='40 \nCopyright 2024. Samsung SDS Co., Ltd. All rights reserved. \n[References] \nBetsy Beyer, Chris Johnes, Jennifer Petoff, Nia ll Richard Murphy, Site Reliability Engineering, \nO’Reilly, 2018 \nTraining Institute, Shin Dong -hyuk, Park Na -ryong, ISMS -P Certification Practical Guide \nConsidering Cloud Environment, Acorn, 2024 \nTTA, Information System Failure Management Guidelines, 2007')

In [ ]:
# 문서를 분할하세요.
# 조각 크기는 400, overlap은 100으로 설정하세요.
# 분할된 문서의 수를 출력하세요.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=100)
split_docs = text_splitter.split_documents(docs)
len(split_docs)

365

In [ ]:
# 임베딩 모델을 생성하세요
# text-embedding-3-small을 사용하세요.
# 임베딩 모델의 동작을 확인하세요.
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings.embed_query('안녕')

[0.0109100341796875,
 -0.0628662109375,
 -0.006488800048828125,
 0.020965576171875,
 0.0123443603515625,
 -0.051544189453125,
 -0.04388427734375,
 0.0234375,
 -0.0295562744140625,
 -0.020965576171875,
 -0.01421356201171875,
 0.0032501220703125,
 0.0203704833984375,
 -0.0028705596923828125,
 0.005870819091796875,
 0.041168212890625,
 -0.0684814453125,
 0.026519775390625,
 0.004364013671875,
 0.02386474609375,
 0.01268768310546875,
 -0.004032135009765625,
 -0.0215606689453125,
 -0.020050048828125,
 0.0241546630859375,
 0.033660888671875,
 0.04254150390625,
 0.004268646240234375,
 0.01248931884765625,
 -0.052490234375,
 0.008392333984375,
 -0.043975830078125,
 -0.0179595947265625,
 0.004886627197265625,
 -0.01540374755859375,
 0.049102783203125,
 0.0024700164794921875,
 -0.037872314453125,
 -0.0218963623046875,
 -0.058563232421875,
 -0.0259552001953125,
 -0.0146484375,
 0.0020809173583984375,
 0.064453125,
 0.06890869140625,
 0.0064697265625,
 -0.059967041015625,
 -0.008453369140625,
 0.0

In [ ]:
# Chroma 벡터 저장소를 생성하여 db 변수에 저장하세요.
from langchain_chroma import Chroma

# Chroma 벡터 저장소 생성
db = Chroma.from_documents(documents=split_docs,
                          embedding=embeddings, collection_name='ws_help_desk')
db

In [ ]:
# 벡터 저장소의 문서 저장소 ID를 출력하세요.
db.get()['ids']

['733ee6b1-2be3-4fc6-b972-fb57be311b92',
 'a50e54e7-9af2-444e-a154-28d9e4482349',
 'efbb6262-9d19-43e6-99f7-a875225e319b',
 'e91c1866-93e3-4b6f-9cb7-d6064a475707',
 '17c36049-69da-48ba-b71a-96670947b036',
 '2e866955-3f7d-4e09-a457-1eca2da186fc',
 'c6232a02-45ac-477f-8943-0cf97fc9094f',
 'e2f75242-c475-4daa-a00b-ad680e26381f',
 '8f099482-f6f8-4c06-93fb-2902f0e08b55',
 'bd4ae341-e636-44ed-8b92-e0b12880898b',
 'c1369c97-ec71-43cd-bf69-7ba676471b07',
 '863079c7-8771-473c-9fc4-2ab260497c3b',
 'd4f9e4f2-d40d-483e-8ace-b976dc0fd91c',
 '6fae3255-e9d0-4c0f-af38-ff7ffd3fdd16',
 'dda9cd9d-dfe8-4761-923c-7ea3f2555044',
 'eb88048d-8e20-4067-9f1b-6dec8385f814',
 '1d6c4951-6a79-4361-8775-70b89867c777',
 'd6e4b7ea-8688-4b91-83dd-7ae2fc91a3b6',
 '01b05858-3803-4318-89e4-f461f23c0c30',
 'c4baa8c3-7955-4c4f-9ae8-73c5331f8cea',
 'aa369625-0436-4f39-9fb3-ac1831e80182',
 '71e5f59e-9021-4063-b02d-267d79c7150a',
 'ced6ff98-0b1b-408e-a524-2378f2a1fa8b',
 '5b59453d-aa26-4605-8dae-898a042bb867',
 '77e9f6a9-584d-

In [ ]:
# 벡터 저장소를 검색기로 변환하여 simple_retriever 변수에 저장하세요.
# k는 5로 설정하세요.
# 검색기의 동작을 확인하세요.
simple_retriever = db.as_retriever(search_kwargs={'k': 10})
simple_retriever.invoke('클라우드 신뢰성')

[Document(id='e2f75242-c475-4daa-a00b-ad680e26381f', metadata={'producer': 'Microsoft® Word 2016', 'source': 'data/sds_cloud_reliability.pdf', 'page_label': '4', 'moddate': '2024-11-12T10:09:15+09:00', 'author': '테크', 'creator': 'Microsoft® Word 2016', 'page': 3, 'creationdate': '2024-11-12T10:09:15+09:00', 'total_pages': 40}, page_content='1 \nCopyright 2024. Samsung SDS Co., Ltd. All rights reserved. \nReliability design principle deals with the ability to minimize data loss and quickly resume \nservices in a failure or disaster situation. If the aforementioned availability design principles are \npre-preparations for automatic fault response (fail-over) through high-availability designs such as'),
 Document(id='c1376a12-d4db-4d85-9f29-a9fc9efe2d5c', metadata={'page': 42, 'creator': 'Microsoft® Word 2016', 'total_pages': 43, 'source': 'data/sds_cloud_operational_excellence.pdf', 'moddate': '2025-08-06T01:51:29+09:00', 'author': '테크', 'page_label': '43', 'creationdate': '2025-08-06T01

In [ ]:
# 요청 재작성 (Query rewrite) 기법을 적용해보세요.
from typing import TypedDict
from langchain_core.runnables import RunnableLambda

class Schema(TypedDict):
    '''재작성된 query를 반환하기 위한 스키마'''
    rewrited_query: str

def query_rewrite(query: str) -> str:
    '''재작성된 query를 반환하는 함수'''

    system_prompt = '''주어진 질문을 벡터 검색에 최적화된 형태로 재작성하세요.
    # 재작성 지침:
    1. 질문의 핵심 의도를 파악하세요
    2. 명확하고 구체적인 키워드를 포함하세요
    3. 검색에 도움이 되는 관련 용어를 추가하세요
    4. 원래 질문과 같은 언어로 작성하세요'''

    llm_rewrite = llm.with_structured_output(Schema) # 스키마 적용
    msgs = [('system', system_prompt), ('user', query)] # chat messages 형태로 전달
    return llm_rewrite.invoke(msgs)['rewrited_query']

rewriter = RunnableLambda(query_rewrite)
rewriter.invoke('클라우드 신뢰성')

'클라우드 신뢰성(Cloud Reliability) 정의, 핵심 개념, 측정 지표(SLI, SLO, SLA), 장애 대응, 가용성, 내결함성, 복원력, 클라우드 인프라 안정성'

In [ ]:
# 프롬프트 템플릿을 자유롭게 생성해보세요.
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """너는 SDS Cloud 기술 전문가야. 아래 제공된 문맥(Context)를 바탕으로 사용자 질문(Question)에 답변하고 근거가 되는 문서의 이름과 페이지를 명시해줘.
답변은 한국어로 해주고, 따옴표('')로 감싸진 영어 용어는 번역하지 말고 그대로 써줘.
문서에서 관련 내용을 찾을 없다면 모른다고 답해줘.

#Question:
{question}

#Context:
{context}

#Answer:"""
)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="너는 SDS Cloud 기술 전문가야. 아래 제공된 문맥(Context)를 바탕으로 사용자 질문(Question)에 답변하고 근거가 되는 문서의 이름과 페이지를 명시해줘.\n답변은 한국어로 해주고, 따옴표('')로 감싸진 영어 용어는 번역하지 말고 그대로 써줘.\n문서에서 관련 내용을 찾을 없다면 모른다고 답해줘.\n\n#Question:\n{question}\n\n#Context:\n{context}\n\n#Answer:")

In [ ]:
# RAG 체인을 생성하세요.
# simple_retriever, query_write_rag_chain의 체인을 각각 생성하세요.
# 출력 파서로 StrOutputParser를 사용하세요.
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

simple_rag_chain = (
    {"context": simple_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query_write_rag_chain = (
    {"context": rewriter | simple_retriever , "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 일반 llm의 응답을 확인하세요.
# 입력은
# '삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?'
# '삼성 sds 클라우드 플랫폼에서 운영 플래닝은 어떻게 설계해야 하나?'
chain = (llm | StrOutputParser())
print(chain.invoke('삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?'))
print(chain.invoke('삼성 sds 클라우드 플랫폼에서 운영 플래닝은 어떻게 설계해야 하나?'))

삼성 SDS 클라우드 플랫폼에서 **신뢰성(reliability)**을 보장하려면, 한마디로 **장애가 나도 서비스가 계속되거나, 빠르게 복구되도록 설계·운영**해야 합니다.  
핵심은 “인프라 이중화 + 자동복구 + 운영관리 + 검증”입니다.

## 1) 이중화 구조로 설계
- **서버 이중화**: 동일 기능의 서버를 2대 이상 구성
- **가용 영역(AZ) 분산**: 한 데이터센터/존 장애가 전체 장애로 이어지지 않게 배치
- **DB 이중화/복제**: Primary-Standby, Multi-AZ, Replica 구성
- **로드밸런서 사용**: 특정 인스턴스 장애 시 자동으로 다른 서버로 분산

## 2) 자동 복구 기능 적용
- **헬스체크**로 비정상 인스턴스 자동 감지
- **오토스케일링**으로 트래픽 증가 시 자동 확장
- **자동 재시작/재배치** 정책 적용
- 컨테이너 환경이면 **Kubernetes의 self-healing** 활용

## 3) 백업과 재해복구(DR) 체계 구축
- **정기 백업**: DB, 파일, 설정값, 이미지 등
- **복구 테스트**: 백업이 실제 복원 가능한지 주기적으로 검증
- **DR 센터/다중 리전 구성**: 대규모 장애 시 대체 환경으로 전환
- **RTO/RPO 정의**:  
  - RTO: 복구까지 허용 시간  
  - RPO: 허용 가능한 데이터 손실량

## 4) 모니터링과 알림 강화
- **CPU, 메모리, 디스크, 네트워크, 응답시간** 모니터링
- **애플리케이션 로그/에러 추적** 수집
- 임계치 초과 시 **실시간 알림**
- 장애 징후를 사전에 탐지하는 **AIOps/지능형 분석** 적용 가능

## 5) 배포 안정성 확보
- **블루-그린 배포** 또는 **카나리 배포**
- 배포 전 **테스트/스테이징 환경 검증**
- 롤백 가능한 배포 방식 사용
- 형상관리와 버전관리 철저히

## 6) 보안과 신뢰성 함께 관리
- 인증/권한 오류도 서비스 장애 원인이므로 **IAM 최소권한** 적용


In [ ]:
# simple RAG의 응답을 확인하세요.
print(simple_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?'))
print(simple_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 운영 플래닝은 어떻게 설계해야 하나?'))

삼성 SDS 클라우드 플랫폼에서 신뢰성을 보장하려면, **장애나 재해가 발생해도 데이터 손실을 최소화하고 서비스를 신속히 복구할 수 있도록 'Failure Recovery Plan'을 수립하고 자동화해야 합니다.**  
구체적으로는 다음이 중요합니다.

1. **Backup 정책의 구성 및 자동화**  
   - 원본 데이터 손상/유실에 대비해 데이터를 별도이고 안전한 저장소에 복사해 두어야 합니다.  
   - 문서에서는 Backup이 가장 일반적인 데이터 보호 수단이라고 설명합니다.

2. **장애 복구 계획 수립**  
   - 서버 및 데이터 손실 상황을 대비해 복구 절차를 마련해야 합니다.

3. **'Cross-region Data Replication' 기반 DR 적용**  
   - 지역 간 데이터 복제를 통해 재해 복구를 지원할 수 있습니다.  
   - 특히 Virtual Server DR 같은 기능을 활용할 수 있습니다.

4. **복구 절차 테스트 및 자동화**  
   - DR 사이트 전환/복귀 절차를 정기적으로 점검하고, 배포 과정을 자동화해 재해 시 오류를 줄여야 합니다.

근거 문서 및 페이지:
- **Samsung Cloud Platform Reliability**: p.11, p.20, p.27, p.34  
  - Backup 정책: p.11  
  - Failure Recovery Plan 수립: p.20  
  - Cross-region Data Replication for Disaster Recovery: p.27  
  - 자동화 및 Failure/Disaster Response Testing: p.34

원하시면 제가 이 내용을 **“신뢰성 보장 체크리스트” 형태로 5줄 요약**해드릴게요.
삼성 SDS 클라우드 플랫폼에서 **운영 플래닝**은 운영을 사람 중심의 수작업에서 벗어나 **자동화 중심으로 설계**하고, 이를 위해 **운영 모델과 배포 전략을 함께 정립**하는 방식으로 접근해야 합니다. 문맥 기준으로는 다음과

In [ ]:
# multi vector RAG의 응답을 확인하세요.
print(query_write_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?'))
print(query_write_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 운영 플래닝은 어떻게 설계해야 하나?'))

삼성 SDS Cloud 플랫폼에서 **신뢰성(reliability)** 을 보장하려면, 문서상으로는 다음과 같은 방법이 핵심입니다.

1. **장애나 재해 상황에서 데이터 손실을 최소화하고 서비스를 빠르게 복구할 수 있도록 설계**해야 합니다.  
2. **백업 정책을 구성하고 자동화**해야 합니다. 백업은 원본 데이터가 서버 장애나 손실로 손상될 때를 대비해, 데이터를 별도의 안전한 저장소에 복사해 두는 가장 일반적인 데이터 보호 방법입니다.  
3. **장애 복구 계획(Failure Recovery Plan)** 을 수립해야 합니다. 서버 및 데이터 손실에 대비한 복구 절차를 마련하는 것이 필요합니다.  

근거 문서:
- **data/sds_cloud_reliability.pdf**
  - **페이지 1**: 신뢰성 설계 원칙이 데이터 손실 최소화와 장애/재해 시 서비스의 신속한 재개를 목표로 한다고 설명
  - **페이지 11**: `'Backup'`은 데이터 보호의 가장 일반적인 방법이며 별도 안전 저장소에 복사하는 방식이라고 설명
  - **페이지 20**: 장애 시 서버 및 데이터 손실에 대비한 장애 복구 계획 수립을 설명

원하시면 제가 이 내용을 바탕으로 **SCP 환경에서의 신뢰성 확보 체크리스트** 형태로도 정리해드릴게요.
제공된 문맥에서는 **‘운영 플래닝’**이라는 용어 자체에 대한 직접적인 정의는 찾기 어렵습니다. 다만 운영 관점에서 설계해야 할 내용은 다음처럼 정리할 수 있습니다.

1. **운영 범위를 먼저 정의**해야 합니다.  
   클라우드 운영에는 서비스 지원, 자원 관리, 성능/장애 관리, 변경 관리, 보안 관리 등이 포함됩니다.

2. **용량과 사용량을 사전에 점검**해야 합니다.  
   Samsung Cloud Platform도 자원 용량 제한이 있으므로, 저장소/컴퓨팅 등의 한계를 확인하고 사용량을 분석해 운영 계획에 반영해야 합니다.

3. **성능과 장애 대응 체계를 마련**해야 합니다.  
   성능 및 용량

In [ ]:
# 재순위화(Re-ranking)을 적용하여 다시 생성해보세요.
# re-ranker는 BAAI/bge-reranker-v2-m3를 사용하세요.
# 기본 검색기를 사용하세요.
# 기본 검색기의 k를 10으로 변경하세요.
# 최종 선택은 5개로 설정하세요.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

# 1차로 50개 추출
retriever = db.as_retriever(search_kwargs={'k': 10})

# 크로스 인코더 모델 생성
model = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-v2-m3')
# 리랭커 생성
compressor = CrossEncoderReranker(model=model, top_n=5)

# base 검색기와 리랭커 결합
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# re-ranker의 동작을 확인하세요.
compression_retriever.invoke('삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?')

[Document(id='e2f75242-c475-4daa-a00b-ad680e26381f', metadata={'page': 3, 'producer': 'Microsoft® Word 2016', 'source': 'data/sds_cloud_reliability.pdf', 'page_label': '4', 'total_pages': 40, 'author': '테크', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-11-12T10:09:15+09:00', 'moddate': '2024-11-12T10:09:15+09:00'}, page_content='1 \nCopyright 2024. Samsung SDS Co., Ltd. All rights reserved. \nReliability design principle deals with the ability to minimize data loss and quickly resume \nservices in a failure or disaster situation. If the aforementioned availability design principles are \npre-preparations for automatic fault response (fail-over) through high-availability designs such as'),
 Document(id='8be43b40-6bf7-420d-afb3-0c1074741634', metadata={'creator': 'Microsoft® Word 2016', 'total_pages': 43, 'page_label': '30', 'source': 'data/sds_cloud_operational_excellence.pdf', 'author': '테크', 'producer': 'Microsoft® Word 2016', 'creationdate': '2025-08-06T01:51:29+09:00', '

In [ ]:
# 재순위화가 적용된 체인을 생성하고 응답을 확인하세요.
re_ranking_rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(re_ranking_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 신뢰성을 보장하려면?'))
print(re_ranking_rag_chain.invoke('삼성 sds 클라우드 플랫폼에서 운영 플래닝은 어떻게 설계해야 하나?'))

삼성 SDS Cloud Platform에서 **신뢰성(Reliability)** 을 보장하려면, **장애나 재해 상황에서 데이터 손실을 최소화하고 서비스를 빠르게 복구할 수 있도록 설계**해야 합니다.  
즉, **고가용성(High Availability) 기반의 자동 장애 전환(fail-over)** 과 **재해 복구(Disaster Recovery, DR)** 를 함께 고려하는 것이 핵심입니다.

문맥에서 확인되는 신뢰성 확보 방법은 다음과 같습니다.

1. **고가용성 설계로 자동 장애 전환 준비**
   - 장애 발생 시 서비스가 자동으로 다른 자원으로 전환되도록 설계합니다.
2. **Cross-region Data Replication for Disaster Recovery**
   - 여러 수준의 storage replication을 활용해 DR을 지원합니다.
3. **Virtual Server DR 활용**
   - virtual servers와 연결된 block storage를 복제하여 재해 복구를 지원합니다.

근거 문서:
- **sds_cloud_reliability.pdf**
  - **page 4**
  - **page 30**

참고로, 문맥상 신뢰성의 핵심 정의는 다음처럼 제시됩니다:  
“ability to minimize data loss and quickly resume services in a failure or disaster situation”

원하시면 제가 위 내용을 바탕으로 **SDS Cloud Platform 신뢰성 보장 방안**을 더 실무적으로 정리해드릴 수 있습니다.
문서에서 확인되는 범위 내에서는, 삼성 SDS 클라우드 플랫폼의 **운영 플래닝**은 크게 다음처럼 설계해야 합니다.

1. **운영 모델을 클라우드에 맞게 재정의**
   - 기존의 인프라/개발 분리 구조가 아니라, **DevOps Team** 중심으로 운영 체계를 바꾸는 방향이 제시됩니다.
   - 즉, 운영과 개발을 분리하기보다 함께 책임지는 구